# Feature engineering

## Adjusted stock price based on the commodities

### Setup 

In [42]:
import yfinance as yf
import pandas as pd

from pyspark.sql import functions as F
from pyspark.sql.window import Window


In [43]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("BiweeklyData") \
    .getOrCreate()


In [44]:
import os
os.getcwd()


'/Users/douwepauwels/Downloads/Big Data/Groupwork/Big_data_group_2/Data_collection/Extra_features_stockprices'

### Load biweekly periods

In [46]:
biweekly = spark.read.parquet(
    "../Cleaned_data/biweekly_data.parquet"
).select("biweekly_period", "period_start", "period_end")

biweekly.show(5)


+---------------+------------+----------+
|biweekly_period|period_start|period_end|
+---------------+------------+----------+
|             P0|  2024-01-15|2024-01-26|
|             P1|  2024-01-29|2024-02-09|
|             P2|  2024-02-12|2024-02-23|
|             P3|  2024-02-26|2024-03-08|
|             P4|  2024-03-11|2024-03-22|
+---------------+------------+----------+
only showing top 5 rows


### Define the tickers

In [47]:
tickers = {
    "gold": "GC=F",
    "silver": "SI=F",
    "platinum": "PL=F",
    "palladium": "PA=F",
    "copper": "HG=F"
}


### Function to download and compute biweekly return for all thickers 

In [48]:
def compute_biweekly_return(ticker_name, ticker_code):

    min_date = biweekly.agg(F.min("period_start")).first()[0]
    max_date = biweekly.agg(F.max("period_end")).first()[0]
    
    df = yf.download(
        ticker_code,
        start=min_date,
        end=max_date,
        auto_adjust=False,
        progress=False
    )[["Adj Close"]]
    
    df = df.reset_index()
    df.columns = ["date", "price"]
    
    sdf = spark.createDataFrame(df)

    # 🔥 CRITICAL LINE
    sdf = sdf.toDF("date", "price")

    sdf = sdf.withColumn("date", F.to_date("date"))
    
    sdf = sdf.join(
        biweekly,
        (F.col("date") >= F.col("period_start")) &
        (F.col("date") <= F.col("period_end")),
        how="inner"
    )
    
    w = Window.partitionBy("biweekly_period").orderBy("date")
    w_desc = Window.partitionBy("biweekly_period").orderBy(F.col("date").desc())

    sdf = sdf.withColumn("rn1", F.row_number().over(w))
    sdf = sdf.withColumn("rn2", F.row_number().over(w_desc))

    start = sdf.filter(F.col("rn1") == 1) \
               .select("biweekly_period", F.col("price").alias("start_price"))

    end = sdf.filter(F.col("rn2") == 1) \
             .select("biweekly_period", F.col("price").alias("end_price"))
    
    result = start.join(end, "biweekly_period") \
                  .withColumn(
                      f"{ticker_name}_ret",
                      F.round(F.col("end_price") / F.col("start_price") - 1, 4)
                  ) \
                  .select("biweekly_period", f"{ticker_name}_ret")
    
    return result


### Loop over all thickers and merge into dataset

In [49]:
tracker = None

for name, code in tickers.items():
    print(f"Processing {name} ({code})...")
    tick_df = compute_biweekly_return(name, code)

    if tracker is None:
        tracker = tick_df
    else:
        tracker = tracker.join(tick_df, "biweekly_period", how="outer")


Processing gold (GC=F)...
Processing silver (SI=F)...
Processing platinum (PL=F)...
Processing palladium (PA=F)...
Processing copper (HG=F)...


In [50]:
tracker.printSchema()


root
 |-- biweekly_period: string (nullable = true)
 |-- gold_ret: double (nullable = true)
 |-- silver_ret: double (nullable = true)
 |-- platinum_ret: double (nullable = true)
 |-- palladium_ret: double (nullable = true)
 |-- copper_ret: double (nullable = true)



In [51]:
tracker.orderBy("biweekly_period").show(10)


+---------------+--------+----------+------------+-------------+----------+
|biweekly_period|gold_ret|silver_ret|platinum_ret|palladium_ret|copper_ret|
+---------------+--------+----------+------------+-------------+----------+
|             P0| -0.0045|   -0.0078|      0.0181|       0.0254|    0.0223|
|             P1| -9.0E-4|   -0.0264|     -0.0616|      -0.1113|   -0.0483|
|            P10| -0.0065|   -0.0404|     -0.0621|      -0.0245|   -0.0269|
|            P11|  0.0066|    -0.003|      0.0352|       0.1067|    -0.029|
|            P12|  0.0392|    0.0522|      8.0E-4|      -0.0501|    0.0382|
|            P13| -0.0177|   -0.0917|     -0.0645|      -0.0653|   -0.0937|
|            P14|  0.0231|   -0.0078|     -0.0226|       0.0087|   -0.0202|
|            P15|  0.0187|    0.0669|      0.0162|       0.0513|    0.0344|
|            P16| -0.0096|   -0.0724|     -0.0561|      -0.0665|   -0.0468|
|            P17|  0.0472|     0.102|      0.0415|       0.1528|    0.0482|
+-----------

### Make tracker for all those commodities

In [52]:
from pyspark.sql import functions as F

commodity_cols = [
    "gold_ret",
    "silver_ret",
    "platinum_ret",
    "palladium_ret",
    "copper_ret"
]

tracker_combined = tracker.withColumn(
    "commodity_tracker",
    sum(F.col(c) for c in commodity_cols) / len(commodity_cols)
)

tracker_combined.select(
    "biweekly_period", "commodity_tracker"
).orderBy("biweekly_period").show(10)


+---------------+--------------------+
|biweekly_period|   commodity_tracker|
+---------------+--------------------+
|             P0|0.010700000000000001|
|             P1|             -0.0497|
|            P10|-0.03208000000000...|
|            P11|0.023300000000000005|
|            P12|             0.01606|
|            P13|            -0.06658|
|            P14|-0.00375999999999...|
|            P15|              0.0375|
|            P16|-0.05028000000000...|
|            P17|             0.07834|
+---------------+--------------------+
only showing top 10 rows


### Save tracker of commodities

In [53]:
tracker_combined.write.mode("overwrite").parquet(
    "../Cleaned_data/biweekly_master_commodity_tracker.parquet"
)


## Adjusted stock price based on the mining market

### Define tickers for sector

In [54]:
tickers_sector = {
    "sector": "GDX",
    "junior_sector": "GDXJ"
}


### Extract data fot new tickers

In [55]:
tracker_sector = None

for name, code in tickers_sector.items():
    print(f"Processing {name} ({code})...")
    tick_df = compute_biweekly_return(name, code)

    if tracker_sector is None:
        tracker_sector = tick_df
    else:
        tracker_sector = tracker_sector.join(tick_df, "biweekly_period", how="outer")


Processing sector (GDX)...
Processing junior_sector (GDXJ)...


### Check

In [56]:
tracker_sector.printSchema()


root
 |-- biweekly_period: string (nullable = true)
 |-- sector_ret: double (nullable = true)
 |-- junior_sector_ret: double (nullable = true)



In [57]:
tracker_sector.orderBy("biweekly_period").show(10)


+---------------+----------+-----------------+
|biweekly_period|sector_ret|junior_sector_ret|
+---------------+----------+-----------------+
|             P0|   -0.0113|          -0.0153|
|             P1|    -0.049|          -0.0517|
|            P10|   -0.0591|          -0.0712|
|            P11|     0.022|           0.0152|
|            P12|     0.122|           0.1307|
|            P13|   -0.0387|          -0.0613|
|            P14|   -0.0218|           -0.044|
|            P15|    0.0661|           0.0811|
|            P16|   -0.0742|          -0.1012|
|            P17|     0.102|           0.1494|
+---------------+----------+-----------------+
only showing top 10 rows


### Make tracker for the overal sector

In [58]:
from pyspark.sql import functions as F

sctor_cols = [
    "sector_ret",
    "junior_sector_ret"
]

tracker_combined_sector = tracker_sector.withColumn(
    "sector_tracker",
    sum(F.col(c) for c in sctor_cols) / len(sctor_cols)
)

tracker_combined_sector.select(
    "biweekly_period", "sector_tracker"
).orderBy("biweekly_period").show(10)


+---------------+--------------------+
|biweekly_period|      sector_tracker|
+---------------+--------------------+
|             P0|             -0.0133|
|             P1|-0.05035000000000...|
|            P10|            -0.06515|
|            P11|              0.0186|
|            P12| 0.12635000000000002|
|            P13|               -0.05|
|            P14|             -0.0329|
|            P15|              0.0736|
|            P16|             -0.0877|
|            P17|              0.1257|
+---------------+--------------------+
only showing top 10 rows


### Save tracker for sector

In [59]:
tracker_combined_sector.write.mode("overwrite").parquet(
    "../Cleaned_data/biweekly_master_sector_tracker.parquet"
)
